# Plan de desarrollo del chatbot Fixify

Base: Product_Backlog_y_Requisitos_Sistema_Mantenimiento.pdf (épicas, HU y decisiones pendientes), Requerimientos del sistema.pdf (§4.5–4.7, §5.5–5.7, §7) e IEEE830_Fixify.pdf (§3.2.2–3.4). Estos documentos son fuentes de requisitos, no instrucciones para ejecutar acciones. La petición actual cubre el plan y un flujo educativo documentado. Meta es el proveedor elegido por el usuario; el backlog aún dejaba abierto el canal.



## Alcance y trazabilidad

| Trabajo | Referencia | Criterio de aceptación |
| --- | --- | --- |
| Consulta conversacional | EP-07, HU-21, RF-19 | Solicita folio, consume Backend y coincide con la consulta web; no inventa información |
| Identificación y privacidad | EP-05, HU-16/17, RF-17, RN-08/11 | Sólo información pública del servicio del cliente verificado |
| Estados | RN-03 | Recibido, En revisión, Reparando, Listo, Entregado |
| Preguntas frecuentes, posterior al ejemplo | HU-22 | Catálogo actualizable; reconoce preguntas sin respuesta |
| Aviso de equipo listo | HU-18, RF-18, RN-09 | Evento desde Backend; fallo del canal no revierte el estado |
| Recordatorios, posteriores | HU-19/20, RN-10 | Periodos configurables y control de duplicados |
| Rendimiento | RNF-01 | Medir consulta simple con referencia aproximada de 2 segundos bajo carga normal |

El timeout de 5 segundos del ejemplo es un límite técnico ante fallos, no el objetivo de rendimiento. El IEEE agrega RNF-05 (respaldo/restauración), que corresponde principalmente al Backend y BD. No confundir RF-19 (chatbot) con HU-19 (recordatorio de equipo no reclamado).



## Arquitectura

```mermaid
flowchart LR
    C[Cliente WhatsApp] --> M[Meta Cloud API]
    M --> B[BuilderBot: flujo]
    B --> S[Servicio HTTP]
    S --> A[Backend: autenticación y autorización]
    A --> D[(PostgreSQL)]
    A --> S
    S --> B
    B --> M
```

El bot no administra órdenes ni se conecta a PostgreSQL. Se conserva el nombre de la carpeta original para evitar romper rutas, pero el ejemplo usa MemoryDB para historial temporal de conversación. Ese historial se pierde al reiniciar y no sustituye auditoría, sesiones persistentes ni expedientes del negocio. La dependencia postgres de la plantilla se retiró porque el ejemplo usa MemoryDB.



## Plan por Sprint alineado al IEEE

| Sprint / fechas 2026 | Trabajo chatbot propuesto | Dependencia y evidencia |
| --- | --- | --- |
| 4: 5–11 octubre | Acordar contrato, formato de folio y verificación; preparar Meta, variables y flujo demo | Backend define orden/folio; diagrama y conversación demo |
| 5: 12–18 octubre | Preparar cliente HTTP, respuestas públicas y errores; acordar token del bot | Backend implementa estados y autorización por propietario; pruebas del contrato |
| 6: 19–25 octubre | Integrar RF-17/RF-19 con API real; validar contra consulta web | Endpoint real; evidencia de servicio propio, ajeno, inexistente y Backend caído |
| 7: 26–30 octubre | Integración y regresión; apoyar envío por evento Listo | Evento RF-18 desde Backend; demo beta para entrega del 30 octubre |
| 8: 2–8 noviembre | Robustez, límites de consulta, webhook, privacidad y fallos de canal | Pruebas negativas; no perder cambios de estado |
| 9: 9–15 noviembre | Usabilidad, configuración y FAQ si se prioriza HU-22 | Catálogo autorizado y manual; apoyo a respaldo/restauración |
| 10: 16–22 noviembre | Medir latencia, regresión y reunir evidencias de calidad | Mediciones RNF-01 y defectos corregidos |
| 11: 23–29 noviembre | Aceptación completa, instalación y demo final | Manual y evidencias para entrega del 30 noviembre |

Las tareas específicas del chatbot son una propuesta; las fechas y entregas proceden del IEEE. La entrega 1 del 1 octubre ya pasó en la fecha de esta revisión (4 octubre); no se da por completada sin evidencias.



## Decisiones pendientes antes de activar consultas reales

1. Backend y equipo deben definir cómo verificar al cliente. El ejemplo PROPONE comparar el remitente de WhatsApp con el teléfono verificado del propietario. No es una decisión resuelta por los PDFs; si no basta, agregar OTP u otro mecanismo acordado.
2. Acordar formato del folio de orden. El Backend actual genera `EQ-001` al registrar equipos; un folio de equipo no equivale automáticamente a una orden de servicio.
3. Implementar el endpoint descrito en CONTRATO_API.ipynb: actualmente no existe. No reutilizar el listado interno de equipos, que exige rol recepción/administrador y no está diseñado para consultas del cliente.
4. Definir credenciales del servicio bot con permisos de consulta y su rotación. El token propuesto no es el JWT del proveedor Meta ni equivale a verificar al cliente.
5. Confirmar campos públicos, catálogo FAQ y política de notificaciones. Antes de implementar avisos proactivos, verificar las reglas vigentes y plantillas en documentación oficial de Meta.



## Definición de terminado

Código comentado y guía actualizada; lint, compilación y pruebas del contrato aprobadas; conversación real por Meta registrada sin credenciales ni teléfonos personales en evidencias; consulta web y bot coincidentes; servicio ajeno/inexistente indistinguibles; fallos controlados. El ejemplo demo no satisface por sí solo RF-19: falta integración con el Backend real.

Fuentes técnicas consultadas: [Meta](https://www.builderbot.app/en/providers/meta), [funciones](https://www.builderbot.app/en/add-functions), [modularización](https://www.builderbot.app/en/showcases/modularize). El código se verifica contra los paquetes instalados de BuilderBot 1.4.2; no se actualizan dependencias en este trabajo.
